# Aula 20 - Notebook: Codificação de Huffman Aplicada à Telemetria SCADA-Core

**Projeto:** SCADA-Core Automática — GRUPO 4  
**Unidade:** Planta Automatizada de Seleção e Classificação Óptica de Grãos com Separação Pneumática Tripla  
**Engenharia de Automação e Controle** — ECAA08  

---

### 1. Objetivos do Notebook
Neste notebook implementamos a **Codificação de Huffman** (*Prefix-Free Optimal Binary Tree*) aplicada ao fluxo de eventos e alarmes do **SCADA-Core Automática**:
1. Construção da Árvore Binária de Huffman via *Min-Heap* (`heapq`) a partir das frequências reais da planta;
2. Cálculo da **Entropia de Shannon $H(X)$**, do **Comprimento Médio $L_{\text{médio}}$** e da **Eficiência Informacional $\eta$**;
3. Simulação de um lote de $1\,000$ eventos discretos (classificações `KXA-501..503`, pulsos `XS-401` e alarmes `ISA-18.2`), com **codificação e decodificação 100% sem perdas**;
4. Aplicação complementar na compressão de uma sequência textual real de **Logs de Alarmes ISA-18.2** do Historiador.

### 2. Funções Auxiliares de Formatação Tabular ASCII

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

### 3. Implementação da Árvore de Huffman, Codificador e Decodificador

In [2]:
import heapq
import math
import random
from collections import Counter
from typing import Dict, List, Optional, Tuple

class NoHuffman:
    def __init__(self, simbolo: Optional[str], freq: int,
                 esq: Optional["NoHuffman"] = None, dir: Optional["NoHuffman"] = None):
        self.simbolo = simbolo
        self.freq = freq
        self.esq = esq
        self.dir = dir

def construir_arvore_huffman(frequencias: Dict[str, int]) -> NoHuffman:
    if not frequencias:
        raise ValueError("Dicionario de frequencias vazio.")
    heap = []
    tie_breaker = 0
    for simbolo, freq in sorted(frequencias.items()):
        heapq.heappush(heap, (freq, tie_breaker, NoHuffman(simbolo, freq)))
        tie_breaker += 1

    while len(heap) > 1:
        f1, _, no_esq = heapq.heappop(heap)
        f2, _, no_dir = heapq.heappop(heap)
        pai = NoHuffman(None, f1 + f2, no_esq, no_dir)
        heapq.heappush(heap, (pai.freq, tie_breaker, pai))
        tie_breaker += 1

    return heap[0][2]

def gerar_tabela_codigos(no: NoHuffman, prefixo: str = "", tabela: Optional[Dict[str, str]] = None) -> Dict[str, str]:
    if tabela is None:
        tabela = {}
    if no.simbolo is not None:
        tabela[no.simbolo] = prefixo if prefixo else "0"
    else:
        if no.esq is not None:
            gerar_tabela_codigos(no.esq, prefixo + "0", tabela)
        if no.dir is not None:
            gerar_tabela_codigos(no.dir, prefixo + "1", tabela)
    return tabela

def codificar_sequencia(sequencia: List[str], tabela: Dict[str, str]) -> str:
    return "".join(tabela[s] for s in sequencia)

def decodificar_bitstream(bitstream: str, raiz: NoHuffman) -> List[str]:
    if raiz.simbolo is not None:
        return [raiz.simbolo] * len(bitstream)
    resultado = []
    atual = raiz
    for bit in bitstream:
        atual = atual.esq if bit == "0" else atual.dir
        if atual.simbolo is not None:
            resultado.append(atual.simbolo)
            atual = raiz
    return resultado

### 4. Cenário 1: Compressão do Fluxo de Eventos e Classificação da Planta
Aplicamos Huffman ao conjunto de $8$ eventos discretos do SCADA-Core em um lote padrão de $1\,000$ ocorrências.

In [3]:
EVENTOS_PLANTA = {
    "KXA-501 (Cat. A)": 720,
    "KXA-502 (Cat. B)": 150,
    "KXA-503 (Cat. C)": 60,
    "XS-401 (Sync)":    40,
    "ALR_NA (Silo)":    15,
    "ALR_PAL (Ar)":      8,
    "ALR_JI (Motor)":    5,
    "XA-901 (Emerg)":    2,
}

raiz_eventos = construir_arvore_huffman(EVENTOS_PLANTA)
tabela_eventos = gerar_tabela_codigos(raiz_eventos)

# Verificacao da propriedade de prefixo (nenhum codigo e prefixo de outro)
lista_codigos = list(tabela_eventos.values())
for i, c1 in enumerate(lista_codigos):
    for j, c2 in enumerate(lista_codigos):
        if i != j:
            assert not c2.startswith(c1), f"Violacao de prefixo: {c1} vs {c2}"

total_ev = sum(EVENTOS_PLANTA.values())
entropia = -sum((f / total_ev) * math.log2(f / total_ev) for f in EVENTOS_PLANTA.values())
l_medio = sum((EVENTOS_PLANTA[s] / total_ev) * len(tabela_eventos[s]) for s in EVENTOS_PLANTA)
bits_fixo = math.ceil(math.log2(len(EVENTOS_PLANTA)))
economia_pct = (1.0 - l_medio / bits_fixo) * 100.0
eficiencia_pct = (entropia / l_medio) * 100.0

linhas_tabela = []
for s, f in sorted(EVENTOS_PLANTA.items(), key=lambda x: -x[1]):
    cod = tabela_eventos[s]
    linhas_tabela.append({
        "Evento / Tag": s,
        "Freq": f,
        "Prob": f"{f/total_ev:.3f}",
        "Fixo": f"{bits_fixo} bits",
        "Codigo Huffman": cod,
        "Tam": f"{len(cod)} bit(s)",
    })

print("=== TABELA DE CODIGOS DE HUFFMAN (SCADA-CORE) ===")
print(formatar_tabela(linhas_tabela))
print(f"\nEntropia de Shannon H(X) : {entropia:.4f} bits/simbolo")
print(f"Comprimento Medio (Huff) : {l_medio:.4f} bits/simbolo")
print(f"Comprimento Fixo         : {bits_fixo:.4f} bits/simbolo")
print(f"Economia vs Codigo Fixo  : {economia_pct:.2f}%")
print(f"Eficiencia H(X)/L_medio  : {eficiencia_pct:.2f}%")

=== TABELA DE CODIGOS DE HUFFMAN (SCADA-CORE) ===
Evento / Tag     | Freq | Prob  | Fixo   | Codigo Huffman | Tam     
-----------------+------+-------+--------+----------------+---------
KXA-501 (Cat. A) | 720  | 0.720 | 3 bits | 1              | 1 bit(s)
KXA-502 (Cat. B) | 150  | 0.150 | 3 bits | 01             | 2 bit(s)
KXA-503 (Cat. C) | 60   | 0.060 | 3 bits | 000            | 3 bit(s)
XS-401 (Sync)    | 40   | 0.040 | 3 bits | 0011           | 4 bit(s)
ALR_NA (Silo)    | 15   | 0.015 | 3 bits | 00100          | 5 bit(s)
ALR_PAL (Ar)     | 8    | 0.008 | 3 bits | 001011         | 6 bit(s)
ALR_JI (Motor)   | 5    | 0.005 | 3 bits | 0010101        | 7 bit(s)
XA-901 (Emerg)   | 2    | 0.002 | 3 bits | 0010100        | 7 bit(s)

Entropia de Shannon H(X) : 1.3838 bits/simbolo
Comprimento Medio (Huff) : 1.5320 bits/simbolo
Comprimento Fixo         : 3.0000 bits/simbolo
Economia vs Codigo Fixo  : 48.93%
Eficiencia H(X)/L_medio  : 90.33%


### 5. Validação Lossless em Lote Simulado de 1.000 Eventos
Geramos uma sequência embaralhada com exatamente as $1\,000$ ocorrências acima, comprimimos para *bitstream* binário, decodificamos pela árvore e comprovamos igualdade bit a bit (`decodificada == original`).

In [4]:
rng = random.Random(42)
sequencia_lote = []
for simb, freq in EVENTOS_PLANTA.items():
    sequencia_lote.extend([simb] * freq)
rng.shuffle(sequencia_lote)

bitstream = codificar_sequencia(sequencia_lote, tabela_eventos)
decodificada = decodificar_bitstream(bitstream, raiz_eventos)

assert decodificada == sequencia_lote, "Erro: decodificacao nao e identica ao original!"

bits_orig_fixo = len(sequencia_lote) * bits_fixo
bits_huffman = len(bitstream)

print("VALIDACAO LOSSLESS CONCLUIDA:")
print(f"  - Eventos processados       : {len(sequencia_lote)}")
print(f"  - Tamanho em codigo fixo    : {bits_orig_fixo} bits ({bits_orig_fixo/8:.1f} bytes)")
print(f"  - Tamanho comprimido (Huff) : {bits_huffman} bits ({bits_huffman/8:.1f} bytes)")
print(f"  - Primeiros 64 bits gerados : {bitstream[:64]}...")
print(f"  - Decodificacao exata?      : {decodificada == sequencia_lote}")

VALIDACAO LOSSLESS CONCLUIDA:
  - Eventos processados       : 1000
  - Tamanho em codigo fixo    : 3000 bits (375.0 bytes)
  - Tamanho comprimido (Huff) : 1532 bits (191.5 bytes)
  - Primeiros 64 bits gerados : 0110000001110111101111101101110111111110110011011000110111110111...
  - Decodificacao exata?      : True


### 6. Cenário 2: Compressão de Logs Textuais de Alarmes ISA-18.2
Também podemos aplicar Huffman diretamente sobre cadeias de caracteres (ASCII 8 bits) gravadas pelo Historiador Nível 3 do SCADA.

In [5]:
log_isa18 = (
    "[SCADA-CORE][ISA-18.2] OK:KXA-501;OK:KXA-501;OK:KXA-501;OK:KXA-502;"
    "OK:KXA-501;OK:KXA-501;REJ:KXA-503;OK:KXA-501;OK:KXA-501;OK:KXA-501;"
    "ALM:PAL-601=1(PRESSAO_BAIXA);INIB:FY-602,FY-603;ACK:OPERADOR;"
    "ALM:PAL-601=0(NORMAL);OK:KXA-501;OK:KXA-501;OK:KXA-501;STANDBY=0;"
)

freq_chars = dict(Counter(log_isa18))
raiz_log = construir_arvore_huffman(freq_chars)
tabela_log = gerar_tabela_codigos(raiz_log)

bits_log = codificar_sequencia(list(log_isa18), tabela_log)
log_restaurado = "".join(decodificar_bitstream(bits_log, raiz_log))
assert log_restaurado == log_isa18

bits_ascii = len(log_isa18) * 8
economia_log = (1.0 - len(bits_log) / bits_ascii) * 100.0

print("=== COMPRESSAO DE LOG TEXTUAL ISA-18.2 ===")
print(f"Caracteres no log         : {len(log_isa18)} ({len(freq_chars)} simbolos distintos)")
print(f"Tamanho ASCII (8 bits)    : {bits_ascii} bits ({len(log_isa18)} bytes)")
print(f"Tamanho Huffman           : {len(bits_log)} bits ({len(bits_log)/8:.1f} bytes)")
print(f"Taxa de reducao           : {economia_log:.2f}%")
print(f"Integridade 100% lossless : {log_restaurado == log_isa18}")

=== COMPRESSAO DE LOG TEXTUAL ISA-18.2 ===
Caracteres no log         : 260 (38 simbolos distintos)
Tamanho ASCII (8 bits)    : 2080 bits (260 bytes)
Tamanho Huffman           : 1186 bits (148.2 bytes)
Taxa de reducao           : 42.98%
Integridade 100% lossless : True
